# 6.5.2 Mixture of Experts (MoE)

Implement top-k gating with load balancing loss. Visualise expert utilisation and gate weight heatmap.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
n, d, E, k = 64, 16, 8, 2
tokens = rng.standard_normal((n, d))
W_gate = rng.standard_normal((d, E)) * 0.5
logits = tokens @ W_gate

def softmax(x): e=np.exp(x-x.max(-1,keepdims=True)); return e/e.sum(-1,keepdims=True)
def topk_gate(logits, k):
    p = softmax(logits); gate = np.zeros_like(p)
    idx = np.argsort(p,1)[:,-k:]
    for i in range(len(p)): gate[i,idx[i]] = p[i,idx[i]]
    gate /= gate.sum(1,keepdims=True)+1e-10
    return gate, idx

gate, idx = topk_gate(logits, k)
lb = E * np.dot((gate>0).mean(0), gate.mean(0))
counts = np.zeros(E)
for i in range(n): counts[idx[i]] += 1
print(f'Load balance loss: {lb:.4f}')
print(f'Expert counts: {counts.astype(int)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(E), counts, color='steelblue')
axes[0].axhline(n*k/E, color='red', linestyle='--', label='Perfect balance')
axes[0].set(xlabel='Expert', ylabel='Tokens', title=f'Expert Utilisation k={k}'); axes[0].legend(); axes[0].grid(True, axis='y', alpha=0.3)

im = axes[1].imshow(gate[:20].T, cmap='Blues', aspect='auto')
axes[1].set(xlabel='Token', ylabel='Expert', title='Gate Weights (first 20 tokens)')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('output/moe_gating.png')
print('Saved moe_gating.png')